# Hierarchical Ward
        **unsupervised › clustering** · notebook `agglomerative_ward`

        **Kapan dipakai:** Ingin hierarki/dendrogram; Ward meminimalkan varians dalam cluster (mirip KMeans).

        **Jebakan umum:** O(n²) memori; hanya beberapa ribu baris. Tidak bisa memprediksi data baru.

**Data demo:** Wine (3 kultivar, 13 fitur). Label asli (bila ada) hanya dipakai untuk MENGECEK hasil, tidak dilihat algoritma.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
PATH = ROOT / "data_latihan/unsupervised/wine.csv"
LABEL_COL = "cultivar"           # hanya untuk evaluasi; None bila tidak ada label
DROP_COLS = []
K = 3                            # jumlah cluster (untuk metode yang membutuhkannya)
SCALE = True                    # standarisasi fitur sebelum clustering (hampir selalu perlu)
MODEL_KEY = "agglomerative_ward"
OUT = out_dir("unsupervised", "clustering", MODEL_KEY)
SEED = 42; set_seed(SEED)

## 2. Data dan skala
Clustering berbasis jarak peka skala: fitur berskala besar mendominasi bila tidak distandarkan.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.base import clone
from superdb.unsupervised import cluster_scores, k_sweep, bootstrap_stability, cluster_profile
df = pd.read_csv(PATH); y_true = df[LABEL_COL].to_numpy() if LABEL_COL else None
X = df.drop(columns=[c for c in [LABEL_COL, *DROP_COLS] if c]).select_dtypes("number")
Z = StandardScaler().fit_transform(X) if SCALE else X.to_numpy(float)
P2 = PCA(n_components=2, random_state=SEED).fit_transform(Z)       # hanya untuk visualisasi
print(Z.shape, "| label asli:", None if y_true is None else dict(pd.Series(y_true).value_counts()))

## 3. Model
Kode `model` bisa disalin. `Z` adalah data terstandarisasi, `K` jumlah cluster.

In [ ]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering

model = AgglomerativeClustering(n_clusters=K, linkage='ward')
model

## 4. Clustering

In [ ]:
import time; t0 = time.time()
labels = np.asarray(model.fit_predict(Z) if hasattr(model, "fit_predict") else model.fit(Z).predict(Z))
print(f"waktu: {time.time() - t0:.1f} dtk | ukuran cluster:", dict(pd.Series(labels).value_counts().sort_index()))

## 5. Kualitas
- **Silhouette** (tinggi=baik), **Davies-Bouldin** (rendah=baik), **Calinski-Harabasz** (tinggi=baik): tanpa label.
- **ARI/NMI**: terhadap label asli (hanya mungkin di data demo berlabel).
- Noise (label -1) dikecualikan dari skor tanpa label.

In [ ]:
metrics = cluster_scores(Z, labels, y_true); display(show_table(metrics))
if y_true is not None:
    display(pd.crosstab(pd.Series(labels, name="cluster"), pd.Series(y_true, name="label asli")))

## 6. Profil cluster (bahan memberi nama cluster di laporan)

In [ ]:
display(cluster_profile(X, labels).round(2))

## 7. Visualisasi (proyeksi PCA 2D)

In [ ]:
from superdb.unsupervised import plot_embedding
fig, ax = plt.subplots(1, 2 if y_true is not None else 1, figsize=(10 if y_true is not None else 5, 4), squeeze=False)
plot_embedding(P2, labels, "hasil clustering", ax=ax[0, 0])
if y_true is not None: plot_embedding(P2, y_true, "label asli", ax=ax[0, 1])
plt.tight_layout(); plt.savefig(OUT / "clusters.png", dpi=130); plt.show()

## 8. Memilih jumlah cluster (sapuan K)
Cari K di mana kurva silhouette memuncak / inertia menekuk (siku) / BIC minimum. Tidak ada satu kriteria yang pasti benar; gabungkan dengan pengetahuan domain.

In [ ]:
sw = k_sweep(Z, lambda k: clone(model).set_params(**{"n_clusters": k}), range(2, 9), y_true)
display(sw.round(3))
fig, ax = plt.subplots(1, 2, figsize=(9, 3)); sw["silhouette"].plot(marker="o", ax=ax[0], title="silhouette (tinggi = baik)")
(sw["inertia"] if "inertia" in sw else sw["BIC"] if "BIC" in sw else sw["davies_bouldin(kecil=baik)"]).plot(marker="o", ax=ax[1], title="inertia / BIC / DB")
plt.tight_layout(); plt.savefig(OUT / "k_sweep.png", dpi=130); plt.show()

## 9. Stabilitas
Ulangi pada subsampel acak 80%; ARI antar hasil mendekati 1 berarti cluster stabil (tidak kebetulan).

In [ ]:
try:
    stab = bootstrap_stability(Z, lambda: clone(model), n=6, seed=SEED); print(f"stabilitas (ARI antar subsampel): {stab[0]:.3f} ± {stab[1]:.3f}")
except Exception as ex:
    stab = (float("nan"), float("nan")); print("stabilitas tidak dapat dihitung:", type(ex).__name__, ex)
pd.DataFrame({"cluster": labels}).to_csv(OUT / "labels.csv", index=False)
save_json({"metrics": metrics, "stability_ari": stab[0]}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Skor internal tinggi belum berarti cluster berguna: baca profil cluster dan beri nama.
- Bandingkan metode: bentuk cluster menentukan algoritma yang cocok (lihat `clustering_moons` / `clustering_circles` untuk kapan tiap metode gagal).